### ACT 1: Integración SQL–Python y conjunto de trabajo

In [15]:
### Carga al DataFrame de pandas desde la BD usando T-SQL y SQLAlchemy

import os
import sys

# Sube de nivel el directorio para poder importar el módulo config_conexion
sys.path.append(os.path.abspath(".."))

import pandas as pd
from config_conexion import obtener_engine
from sqlalchemy import text

# 1. Conexión a la base de datos
engine = obtener_engine()

# 2. Lectura de la consulta T-SQL desde el archivo
with open("../sql/conjunto_trabajo.sql", "r", encoding="utf-8") as f:
    query_sql = text(f.read())

# 3. Parámetros de filtrado
parametros = {"fecha_inicio": "2005-01-01", "fecha_fin": "2006-12-31"}

# 4. Extracción al DataFrame con pandas.read_sql
df_trabajo = pd.read_sql(query_sql, con=engine, params=parametros)

# Confirmación de carga
print(
    f"Conjunto de trabajo cargado: {df_trabajo.shape[0]} filas y"
    f" {df_trabajo.shape} columnas."
)

# Muestra el DataFrame resultante
df_trabajo.head()

Conjunto de trabajo cargado: 41 filas y (41, 4) columnas.


,fecha,cantidad_alquileres,ingresos_totales,clientes_activos
0,2005-05-24,8,29.92,8
1,2005-05-25,137,573.63,129
2,2005-05-26,174,754.26,155
3,2005-05-27,166,684.34,150
4,2005-05-28,196,804.04,166


### Justificación del Reparto de Responsabilidades (SQL Server y Pandas)

Considerando lo visto en clase y la bibliografía oficial de la U5: 

Tomamos la eficiencia en manejo de grandes volúmenes de datos de SQL Server aprovechando su gran capacidad de optimización, permitiéndonos así, delegarle las tareas de: 

** Unión relacional (LEFT JOIN), para las tablas de interés (alquiler y pago) por medio de sus claves primarias y foráneas. 
** Filtrado (WHERE), delimitándole el rango temporal pasado por parámetros, manteniendo la seguridad y evitando inyecciones. 
** Agregación como GROUP BY y funciones de agregación, minimizando y reduciendo el volumen de datos transferidos. 

Una vez procesada la consulta en la BD, nos apoyamos en la flexibilidad de Pandas el consumo y análisis de datos, al momento de para trabajar con diversas fuentes, como la configuración de conexión con la base y el consumo de scripts TSQL. 

Así también el trabajo de seguridad en conjunto, realizando la otra labor de parámetros, para evitar inyecciones. 
Se definieron placeholders (:fecha_inicio y :fecha_fin) como parámetros, que SQL espera. Desde Pandas se brindan los valores reales mediante 'params' de 'pd.read_sql()', evitando concatenación de cadenas y f-strings. 

En el mismo orden, tomamos su capacidad de transformación para leer datos y cargarlos a un DataFrame, permitiéndonos así, generar una visualización más limpia para un próximo análisis. 


In [ ]:
### Comparación de enfoques: T-SQL vs pandas para las mismas metricas

# Sube de nivel el directorio para poder importar el módulo config_conexion

import os
import sys

sys.path.append(os.path.abspath(".."))

import time
import pandas as pd
from sqlalchemy import text
from config_conexion import obtener_engine
from esquema import ESQUEMA

# Inicialización de la conexión y parámetros de filtrado

engine = obtener_engine()
params = {"fecha_inicio": "2005-01-01", "fecha_fin": "2006-12-31"}

# ==============================================================================
# ENFOQUE A: Íntegramente en T-SQL (Agregación en el motor mediante archivo .sql)
# ==============================================================================
t0_tsql = time.perf_counter()

# Leemos la consulta T-SQL desde el archivo externo
with open("../sql/conjunto_trabajo.sql", "r", encoding="utf-8") as f:
    query_tsql = text(f.read())

df_tsql = pd.read_sql(query_tsql, con=engine, params=params)
tiempo_tsql = time.perf_counter() - t0_tsql

# ==============================================================================
# ENFOQUE B: Íntegramente en pandas sobre datos crudos
# ==============================================================================
t0_pandas = time.perf_counter()

# 1. Extracción de tablas crudas sin fstring - mejoras 
df_alquiler_crudo = pd.read_sql_table(
    table_name="alquiler", 
    schema=ESQUEMA,
    con=engine, 
    columns=["id_alquiler", "fecha_alquiler", "id_cliente"]
)
df_pago_crudo = pd.read_sql_table(
    table_name="pago",
    schema=ESQUEMA,
    con=engine,
    columns=["id_alquiler", "monto"]
)

# 2. Transformación y filtrado previo en pandas
df_alquiler_crudo["fecha_alquiler"] = pd.to_datetime(
    df_alquiler_crudo["fecha_alquiler"]
)
df_alquiler_filtrado = df_alquiler_crudo[
    (df_alquiler_crudo["fecha_alquiler"] >= params["fecha_inicio"])
    & (df_alquiler_crudo["fecha_alquiler"] <= params["fecha_fin"])
].copy()

df_alquiler_filtrado["fecha"] = df_alquiler_filtrado["fecha_alquiler"].dt.date

# 3. Réplica del LEFT JOIN mediante merge
df_merged = df_alquiler_filtrado.merge(df_pago_crudo, on="id_alquiler", how="left")

# 4. Agregación vectorizada en pandas (3 métricas)
df_pandas = (
    df_merged.groupby("fecha", as_index=False)
    .agg(
        cantidad_alquileres=("id_alquiler", "count"),
        ingresos_totales=("monto", lambda x: x.sum(min_count=1)),
        clientes_activos=("id_cliente", "nunique"),
    )
    .sort_values("fecha")
    .reset_index(drop=True)
)

# Rellenar posibles nulos de ingresos con 0.0 (COALESCE)
df_pandas["ingresos_totales"] = df_pandas["ingresos_totales"].fillna(0.0)

tiempo_pandas = time.perf_counter() - t0_pandas

# ==============================================================================
# IMPRESIÓN DE TIEMPOS
# ==============================================================================
print(f" Tiempo Enfoque T-SQL (Agregado): {tiempo_tsql:.4f} segundos")
print(f" Tiempo Enfoque pandas (Crudo):   {tiempo_pandas:.4f} segundos")
print(f" Diferencia de rendimiento:       El Enfoque T-SQL fue"
    f" {tiempo_pandas / tiempo_tsql:.2f}x más rápido.\n"
)

# ==============================================================================
# VERIFICACIÓN DE COINCIDENCIA ESTRICTA
# ==============================================================================
df_tsql["fecha"] = pd.to_datetime(df_tsql["fecha"])
df_pandas["fecha"] = pd.to_datetime(df_pandas["fecha"])
df_tsql["ingresos_totales"] = df_tsql["ingresos_totales"].astype(float)
df_pandas["ingresos_totales"] = df_pandas["ingresos_totales"].astype(float)

try:
    pd.testing.assert_frame_equal(
        df_tsql, df_pandas, check_dtype=False, check_exact=False, rtol=1e-5
    )
    print(
        "¡Éxito! Ambos resultados coinciden exactamente fila por fila y"
        " columna por columna."
    )
except AssertionError as e:
    print("Se detectaron diferencias entre ambos enfoques:")
    print(e)

 Tiempo Enfoque T-SQL (Agregado): 0.1102 segundos
 Tiempo Enfoque pandas (Crudo):   0.7776 segundos
 Diferencia de rendimiento:       El Enfoque T-SQL fue 7.06x más rápido.

¡Éxito! Ambos resultados coinciden exactamente fila por fila y columna por columna.


### Conclusión 

A partir del analisis realizado sobre el cálculo de métricas diarias (cantidad_alquileres, ingresos_totales y clientes_activos), se concluye que:

Coincidencia Exacta: La validación con pd.testing.assert_frame_equal() validó que ambos enfoques producen un resultado idéntico en filas y columnas. Demostrando así, que la lógica de agrupación, filtrado de fechas y tratamiento de nulos implementada en Pandas replica el comportamiento del motor T-SQL.   

Diferencia de Rendimiento: El Enfoque A (T-SQL) completó la ejecución en 0.11 segundos, mientras que el Enfoque B (Pandas) requirió 0.77 segundos. 

Causa: La diferencia de rendimiento (más rápido en T-SQL) radica en que el Enfoque B trajo todas las filas crudas desde SQL Server hacia la memoria RAM de Python antes de filtrar y agrupar. En cambio, el Enfoque A resolvió el filtrado y la agregación internamente en la base de datos, transmitiendo únicamente la matriz final de 41 filas consolidadas a través de la red.

### ACT 2: Perfilado y diagnóstico de calidad de datos

In [34]:
import pandas as pd

# 2.1 RECONOCIMIENTO ESTRUCTURAL Y DE TIPOS 
# ------------------------------------------

print("1. Forma del Conjunto de Trabajo (DataFrame) (shape)")
print(f"Filas: {df_trabajo.shape[0]} | Columnas: {df_trabajo.shape[1]}\n")

print("2. Tipos de Datos de Cada Columna (dtypes)")
print(df_trabajo.dtypes, "\n")  

print("3. Primeras Filas del Conjunto de Trabajo (head)")
print(df_trabajo.head(), "\n")

print("4. Muestra Aleatoria de Filas del Conjunto de Trabajo (sample)")
print(df_trabajo.sample(5, random_state=1))


1. Forma del Conjunto de Trabajo (DataFrame) (shape)
Filas: 41 | Columnas: 4

2. Tipos de Datos de Cada Columna (dtypes)
fecha                  datetime64[s]
cantidad_alquileres            int64
ingresos_totales             float64
clientes_activos               int64
dtype: object 

3. Primeras Filas del Conjunto de Trabajo (head)
       fecha  cantidad_alquileres  ingresos_totales  clientes_activos
0 2005-05-24                    8             29.92                 8
1 2005-05-25                  137            573.63               129
2 2005-05-26                  174            754.26               155
3 2005-05-27                  166            684.34               150
4 2005-05-28                  196            804.04               166 

4. Muestra Aleatoria de Filas del Conjunto de Trabajo (sample)
        fecha  cantidad_alquileres  ingresos_totales  clientes_activos
3  2005-05-27                  166            684.34               150
2  2005-05-26                  174     

In [ ]:
# Revisión de Tipo de Datos

# Columnas que llegaron como texto
columnas_texto = df_trabajo.columns[df_trabajo.dtypes == "object"]
print("Columnas de texto:", list(columnas_texto), "\n")

# Detalles de las columnas de texto
for col in columnas_texto:
    print(f"{col}: {df_trabajo[col].head(3).tolist()}")

Columnas de texto: ['fecha'] 

fecha: [datetime.date(2005, 5, 24), datetime.date(2005, 5, 25), datetime.date(2005, 5, 26)]


In [32]:
# Corrección de Tipo de Datos

df_trabajo["fecha"] = pd.to_datetime(df_trabajo["fecha"], errors="coerce")

print("\nTipos de Datos post corrección (dtypes)")
print(df_trabajo.dtypes)
print("\nFechas no parseables:", df_trabajo["fecha"].isna().sum())


Tipos de Datos post corrección (dtypes)
fecha                  datetime64[s]
cantidad_alquileres            int64
ingresos_totales             float64
clientes_activos               int64
dtype: object

Fechas no parseables: 0


In [33]:
# 2.1.5. Resumen de la estructura del DataFrame (info)
df_trabajo.info()

<class 'pandas.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype        
---  ------               --------------  -----        
 0   fecha                41 non-null     datetime64[s]
 1   cantidad_alquileres  41 non-null     int64        
 2   ingresos_totales     41 non-null     float64      
 3   clientes_activos     41 non-null     int64        
dtypes: datetime64[s](1), float64(1), int64(2)
memory usage: 1.4 KB


In [35]:
# 2.1.6. Estadisticos descriptivos
df_trabajo.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])

,fecha,cantidad_alquileres,ingresos_totales,clientes_activos
count,41,41.000000,41.000000,41.000000
mean,2005-07-13 21:04:23,391.317073,1644.062439,265.756098
min,2005-05-24 00:00:00,8.000000,29.920000,8.000000
1%,2005-05-24 09:36:00,11.200000,41.088000,11.200000
5%,2005-05-26 00:00:00,23.000000,111.770000,23.000000
25%,2005-06-16 00:00:00,174.000000,700.370000,155.000000
50%,2005-07-09 00:00:00,461.000000,1938.390000,319.000000
75%,2005-08-01 00:00:00,621.000000,2577.800000,383.000000
95%,2005-08-22 00:00:00,659.000000,2817.290000,404.000000
99%,2005-12-06 00:00:00,675.800000,2857.190000,409.800000
